In [133]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [171]:
import sys
sys.path.append("..")

from utils import *
import json 
import pandas as pd
import random
from pathlib import Path
import pickle
from transformers import AutoTokenizer
from operator import itemgetter

In [135]:
MODELTYPE = "deepset/gbert-base"
DATASET = "med_indication_all_RF_diag"
DATASETPATH =  Path("../data") / f"ind.{DATASET}"
DATASETFILE = Path("../data") / "medindcls_bert.json"

adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, _, _ = load_corpus(DATASETPATH)
doc_mask = train_mask + val_mask + test_mask
adj.sum()

tokenizer = AutoTokenizer.from_pretrained(MODELTYPE)

if not DATASETFILE.exists():
    print("Creating dataset")
    dataset = CleanClinicDataset(tokenizer=tokenizer, clean=False)
    with open(DATASETFILE, "wb") as f:
        print(f"Saving dataset under {DATASETFILE}")
        pickle.dump(dataset, f)
else:
    print(f"Loading dataset from: {DATASETFILE}")
    with open(DATASETFILE, "rb") as f:
        dataset = pickle.load(f)

27996
Loading dataset from: ../data/medindcls_bert.json


/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator OneHotEncoder from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/tmp/ipykernel_682855/3214339880.py:21: DeprecationWarning: Please use `csr_matrix` from the `scipy.sparse` namespace, the `scipy.sparse.csr` namespace is deprecated.
  dataset = pickle.loa

In [136]:
mixfactor = 0.5
ig_gcn_only_path = f"../models/gcn/{mixfactor}/_ig_attrs_gcn_only.json"
shap_gcn_only_path = f"../models/gcn/{mixfactor}/_shap_values_gcn_only.json"
ig_gcn_bert_path = f"../models/gcn/{mixfactor}/_ig_attrs_gcn_bert.json"
shap_gcn_bert_path = f"../models/gcn/{mixfactor}/_shap_values_gcn_bert.json"

ig_gcn_only_values = pickle.load(open(ig_gcn_only_path, "rb"))
ig_gcn_bert_values = pickle.load(open(ig_gcn_bert_path, "rb"))
shap_gcn_only_values = pickle.load(open(shap_gcn_only_path, "rb"))
shap_gcn_bert_values = pickle.load(open(shap_gcn_bert_path, "rb"))

In [137]:
ig_gcn_only_values[-1]

array([-0.00109209,  0.00087921,  0.00277563, ..., -0.00191309,
        0.0011486 ,  0.34732565])

In [138]:
top_n_interpret = 10
top_ig_gcn_only_values = np.argpartition(ig_gcn_only_values, -top_n_interpret)[:, -top_n_interpret:]
top_ig_gcn_bert_values = np.argpartition(ig_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]
top_shap_gcn_only_values = np.argpartition(shap_gcn_only_values, -top_n_interpret)[:, -top_n_interpret:]
top_shap_gcn_bert_values = np.argpartition(shap_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]
top_ig_gcn_only_values[-1]

array([ 649, 2007, 2200, 2617, 1029, 1183, 1208, 1218, 1778, 2698])

In [139]:
random.seed(0)
idx = np.arange(len(dataset))
random.shuffle(idx)

train_len = 1889
val_len = 270
test_len = 540
nb_word = 25297

train_idx, val_idx, test_idx = (
    idx[: int(len(idx) * 0.7)],
    idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
    idx[int(len(idx) * 0.8) :],
)

def map_to_idx(x):
    if x < train_len:
        return train_idx[x]
    elif x < val_len + train_len:
        return val_idx[x - train_len]
    else:
        return test_idx[x - val_len - train_len]

In [140]:
top_ig_gcn_only_values = np.vectorize(map_to_idx)(top_ig_gcn_only_values)
top_ig_gcn_bert_values = np.vectorize(map_to_idx)(top_ig_gcn_bert_values)
top_shap_gcn_only_values = np.vectorize(map_to_idx)(top_shap_gcn_only_values)
top_shap_gcn_bert_values = np.vectorize(map_to_idx)(top_shap_gcn_bert_values)

In [141]:
ig_gcn_only_df = pd.DataFrame(top_ig_gcn_only_values, index=test_idx)
ig_gcn_bert_df = pd.DataFrame(top_ig_gcn_bert_values, index=test_idx)
shap_gcn_only_df = pd.DataFrame(top_shap_gcn_only_values, index=test_idx)
shap_gcn_bert_df = pd.DataFrame(top_shap_gcn_bert_values, index=test_idx)

In [142]:
ig_gcn_only_df

,0,1,2,3,4,5,6,7,8,9
2298,2284,2286,2282,2297,2298,2293,2294,2292,2291,2300
2130,820,818,819,821,2132,2135,2136,2130,2133,2131
2559,580,582,589,587,2558,2562,2556,2557,2559,2563
1046,2163,2164,123,1380,1378,2165,1379,122,125,124
1991,1254,1719,1357,1354,1990,1351,1991,1355,1718,1356
...,...,...,...,...,...,...,...,...,...,...
2094,1441,2511,294,292,291,2094,293,1337,1335,1334
1060,1466,1056,1474,1061,1058,1055,1057,1060,1059,1054
165,2149,620,2551,675,1912,1140,676,165,166,619
1722,1731,1734,1721,1728,1730,1735,1736,1726,1722,1732


In [143]:
ig_gcn_only_df = pd.melt(ig_gcn_only_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")
ig_gcn_bert_df = pd.melt(ig_gcn_bert_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")
shap_gcn_only_df = pd.melt(shap_gcn_only_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")
shap_gcn_bert_df = pd.melt(shap_gcn_bert_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")

In [144]:
labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_only_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_only_df.rel_id])
ig_gcn_only_df["label"] = labels
ig_gcn_only_df["rel_label"] = rel_labels

labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_bert_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_bert_df.rel_id])
ig_gcn_bert_df["label"] = labels
ig_gcn_bert_df["rel_label"] = rel_labels

labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_only_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_only_df.rel_id])
shap_gcn_only_df["label"] = labels
shap_gcn_only_df["rel_label"] = rel_labels

labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_bert_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_bert_df.rel_id])
shap_gcn_bert_df["label"] = labels
shap_gcn_bert_df["rel_label"] = rel_labels

In [145]:
ig_gcn_only_df

,id,rel_id,label,rel_label
0,2298,2284,Blutdrucksenker_beides,Blutdrucksenker_beides
1,2130,820,Blutdrucksenker_Herzschw,Blutdrucksenker_Herzschw
2,2559,580,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck
3,1046,2163,Cholesterinsenker_unklar,DM_Insulin und Tabletten
4,1991,1254,Blutdrucksenker_unklar,Blutdrucksenker_unklar
...,...,...,...,...
5395,2094,1334,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck
5396,1060,1054,Blutdrucksenker_beides,Blutdrucksenker_beides
5397,165,619,Cholesterinsenker_beides,Cholesterinsenker_beides
5398,1722,1732,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck


In [146]:
ig_gcn_only_source_df = ig_gcn_only_df[["id", "label"]].drop_duplicates()
ig_gcn_only_target_df = ig_gcn_only_df[["rel_id", "rel_label"]].drop_duplicates()
ig_gcn_only_node_df = pd.concat([ig_gcn_only_source_df, ig_gcn_only_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

ig_gcn_bert_source_df = ig_gcn_bert_df[["id", "label"]].drop_duplicates()
ig_gcn_bert_target_df = ig_gcn_bert_df[["rel_id", "rel_label"]].drop_duplicates()
ig_gcn_bert_node_df = pd.concat([ig_gcn_bert_source_df, ig_gcn_bert_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

shap_gcn_only_source_df = shap_gcn_only_df[["id", "label"]].drop_duplicates()
shap_gcn_only_target_df = shap_gcn_only_df[["rel_id", "rel_label"]].drop_duplicates()
shap_gcn_only_node_df = pd.concat([shap_gcn_only_source_df, shap_gcn_only_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

shap_gcn_bert_source_df = shap_gcn_bert_df[["id", "label"]].drop_duplicates()
shap_gcn_bert_target_df = shap_gcn_bert_df[["rel_id", "rel_label"]].drop_duplicates()
shap_gcn_bert_node_df = pd.concat([shap_gcn_bert_source_df, shap_gcn_bert_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

In [177]:
ig_gcn_only_df.rel_id.value_counts()

1370    14
1912    13
1371    13
1372    11
2165    10
        ..
1326     1
1907     1
1684     1
575      1
1054     1
Name: rel_id, Length: 2201, dtype: int64

In [147]:
ig_gcn_only_G=nx.from_pandas_edgelist(ig_gcn_only_df, "id", 'rel_id')#, create_using=nx.DiGraph)
shap_gcn_only_G=nx.from_pandas_edgelist(shap_gcn_only_df, "id", 'rel_id')#, create_using=nx.DiGraph)

ig_gcn_bert_G=nx.from_pandas_edgelist(ig_gcn_bert_df, "id", 'rel_id')#, create_using=nx.DiGraph)
shap_gcn_bert_G=nx.from_pandas_edgelist(shap_gcn_bert_df, "id", 'rel_id')#, create_using=nx.DiGraph)

In [154]:
ig_gcn_only_id_df = ig_gcn_only_df[["id", "label"]]
ig_gcn_only_rel_df = ig_gcn_only_df[["rel_id", "rel_label"]]
ig_gcn_bert_id_df = ig_gcn_bert_df[["id", "label"]]
ig_gcn_bert_rel_df = ig_gcn_bert_df[["rel_id", "rel_label"]]
shap_gcn_only_id_df = shap_gcn_only_df[["id", "label"]]
shap_gcn_only_rel_df = shap_gcn_only_df[["rel_id", "rel_label"]]
shap_gcn_bert_id_df = shap_gcn_bert_df[["id", "label"]]
shap_gcn_bert_rel_df = shap_gcn_bert_df[["rel_id", "rel_label"]]

In [155]:
new_columns = ["id", "label"]
ig_gcn_only_id_df.columns = new_columns
ig_gcn_only_rel_df.columns = new_columns
ig_gcn_bert_id_df.columns = new_columns
ig_gcn_bert_rel_df.columns = new_columns
shap_gcn_only_id_df.columns = new_columns
shap_gcn_only_rel_df.columns = new_columns
shap_gcn_bert_id_df.columns = new_columns
shap_gcn_bert_rel_df.columns = new_columns

In [156]:
ig_gcn_only_id_rel_df = pd.concat([ig_gcn_only_id_df, ig_gcn_only_rel_df], ignore_index=True).drop_duplicates()
ig_gcn_only_id2label = dict(zip(ig_gcn_only_id_rel_df.id, ig_gcn_only_id_rel_df.label))
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_id2label, "label")

ig_gcn_bert_id_rel_df = pd.concat([ig_gcn_bert_id_df, ig_gcn_bert_rel_df], ignore_index=True).drop_duplicates()
ig_gcn_bert_id2label = dict(zip(ig_gcn_bert_id_rel_df.id, ig_gcn_bert_id_rel_df.label))
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2label, "label")

shap_gcn_only_id_rel_df = pd.concat([shap_gcn_only_id_df, shap_gcn_only_rel_df], ignore_index=True).drop_duplicates()
shap_gcn_only_id2label = dict(zip(shap_gcn_only_id_rel_df.id, shap_gcn_only_id_rel_df.label))
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_id2label, "label")

shap_gcn_bert_id_rel_df = pd.concat([shap_gcn_bert_id_df, shap_gcn_bert_rel_df], ignore_index=True).drop_duplicates()
shap_gcn_bert_id2label = dict(zip(shap_gcn_bert_id_rel_df.id, shap_gcn_bert_id_rel_df.label))
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2label, "label")

In [157]:
ig_gcn_only_id2text = {id: dataset.texts[id] for id in ig_gcn_only_id_rel_df.id}
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_id2text, "text")

ig_gcn_bert_id2text = {id: dataset.texts[id] for id in ig_gcn_bert_id_rel_df.id}
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2text, "text")

shap_gcn_only_id2text = {id: dataset.texts[id] for id in shap_gcn_only_id_rel_df.id}
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_id2text, "text")

shap_gcn_bert_id2text = {id: dataset.texts[id] for id in shap_gcn_bert_id_rel_df.id}
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2text, "text")

In [159]:
ig_gcn_only_id2drug = {node: ig_gcn_only_G.nodes()[node]["text"].split(" ")[1] for node in ig_gcn_only_G.nodes()}
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_id2drug, "drug")

ig_gcn_bert_id2drug = {node: ig_gcn_bert_G.nodes()[node]["text"].split(" ")[1] for node in ig_gcn_bert_G.nodes()}
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2drug, "drug")

shap_gcn_only_id2drug = {node: shap_gcn_only_G.nodes()[node]["text"].split(" ")[1] for node in shap_gcn_only_G.nodes()}
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_id2drug, "drug")

shap_gcn_bert_id2drug = {node: shap_gcn_bert_G.nodes()[node]["text"].split(" ")[1] for node in shap_gcn_bert_G.nodes()}
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2drug, "drug")

In [163]:
(nx.is_connected(ig_gcn_only_G), nx.number_connected_components(ig_gcn_only_G), nx.is_connected(ig_gcn_bert_G), nx.number_connected_components(ig_gcn_bert_G),
 nx.is_connected(shap_gcn_only_G), nx.number_connected_components(shap_gcn_only_G), nx.is_connected(shap_gcn_bert_G), nx.number_connected_components(shap_gcn_bert_G))

(False, 16, False, 12, False, 5, False, 5)

In [168]:
ig_gcn_only_components = nx.connected_components(ig_gcn_only_G)
ig_gcn_only_largest_component = max(ig_gcn_only_components, key=len)
ig_gcn_only_subgraph = ig_gcn_only_G.subgraph(ig_gcn_only_largest_component)

ig_gcn_bert_components = nx.connected_components(ig_gcn_bert_G)
ig_gcn_bert_largest_component = max(ig_gcn_bert_components, key=len)
ig_gcn_bert_subgraph = ig_gcn_bert_G.subgraph(ig_gcn_bert_largest_component)

shap_gcn_only_components = nx.connected_components(shap_gcn_only_G)
shap_gcn_only_largest_component = max(shap_gcn_only_components, key=len)
shap_gcn_only_subgraph = shap_gcn_only_G.subgraph(shap_gcn_only_largest_component)

shap_gcn_bert_components = nx.connected_components(shap_gcn_bert_G)
shap_gcn_bert_largest_component = max(shap_gcn_bert_components, key=len)
shap_gcn_bert_subgraph = shap_gcn_bert_G.subgraph(shap_gcn_bert_largest_component)

nx.diameter(ig_gcn_only_subgraph), nx.diameter(ig_gcn_bert_subgraph), nx.diameter(shap_gcn_only_subgraph), nx.diameter(shap_gcn_bert_subgraph)

(23, 20, 18, 17)

In [169]:
ig_gcn_only_triadic_closure = nx.transitivity(ig_gcn_only_G)
ig_gcn_bert_triadic_closure = nx.transitivity(ig_gcn_bert_G)
shap_gcn_only_triadic_closure = nx.transitivity(shap_gcn_only_G)
shap_gcn_bert_triadic_closure = nx.transitivity(shap_gcn_bert_G)
ig_gcn_only_triadic_closure, ig_gcn_bert_triadic_closure, shap_gcn_only_triadic_closure, shap_gcn_bert_triadic_closure

(0.27452484742807326,
 0.24792648132174375,
 0.20904645476772615,
 0.20619574468085106)

In [172]:
ig_gcn_only_degree_dict = dict(ig_gcn_only_G.degree(ig_gcn_only_G.nodes()))
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_degree_dict, 'degree')
ig_gcn_only_sorted_degree = sorted(ig_gcn_only_degree_dict.items(), key=itemgetter(1), reverse=True)

ig_gcn_bert_degree_dict = dict(ig_gcn_bert_G.degree(ig_gcn_bert_G.nodes()))
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_degree_dict, 'degree')
ig_gcn_bert_sorted_degree = sorted(ig_gcn_bert_degree_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_only_degree_dict = dict(shap_gcn_only_G.degree(shap_gcn_only_G.nodes()))
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_degree_dict, 'degree')
shap_gcn_only_sorted_degree = sorted(shap_gcn_only_degree_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_bert_degree_dict = dict(shap_gcn_bert_G.degree(shap_gcn_bert_G.nodes()))
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_degree_dict, 'degree')
shap_gcn_bert_sorted_degree = sorted(shap_gcn_bert_degree_dict.items(), key=itemgetter(1), reverse=True)

#for node, degree in shap_sorted_degree[:10]:
#    print(shap_G.nodes()[node]["drug"])

ig_gcn_only_sorted_degree[:10], ig_gcn_bert_sorted_degree[:10], shap_gcn_only_sorted_degree[:10], shap_gcn_bert_sorted_degree[:10]

([(1912, 22),
  (1745, 20),
  (905, 18),
  (1372, 18),
  (1354, 17),
  (2164, 17),
  (1928, 16),
  (1746, 16),
  (2456, 16),
  (1380, 16)],
 [(1354, 24),
  (1912, 20),
  (2029, 18),
  (1140, 18),
  (1835, 18),
  (1837, 17),
  (1550, 17),
  (1928, 16),
  (83, 16),
  (628, 16)],
 [(1912, 25),
  (1786, 19),
  (2029, 18),
  (1354, 18),
  (1550, 18),
  (905, 17),
  (1140, 17),
  (254, 17),
  (1630, 16),
  (1783, 16)],
 [(1912, 22),
  (1354, 18),
  (2029, 17),
  (1745, 16),
  (905, 16),
  (1783, 16),
  (1140, 16),
  (1786, 16),
  (2389, 16),
  (1630, 15)])

In [173]:
ig_gcn_only_betweenness_dict = nx.betweenness_centrality(ig_gcn_only_G)
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_betweenness_dict, 'betweenness')
ig_gcn_only_sorted_betweenness = sorted(ig_gcn_only_betweenness_dict.items(), key=itemgetter(1), reverse=True)

ig_gcn_bert_betweenness_dict = nx.betweenness_centrality(ig_gcn_bert_G)
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_betweenness_dict, 'betweenness')
ig_gcn_bert_sorted_betweenness = sorted(ig_gcn_bert_betweenness_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_only_betweenness_dict = nx.betweenness_centrality(shap_gcn_only_G)
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_betweenness_dict, 'betweenness')
shap_gcn_only_sorted_betweenness = sorted(shap_gcn_only_betweenness_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_bert_betweenness_dict = nx.betweenness_centrality(shap_gcn_bert_G)
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_betweenness_dict, 'betweenness')
shap_gcn_bert_sorted_betweenness = sorted(shap_gcn_bert_betweenness_dict.items(), key=itemgetter(1), reverse=True)


#for node, degree in shap_sorted_betweenness[:10]:
#    print(shap_G.nodes()[node]["drug"])
#for node, degree in ig_sorted_betweenness[:10]:
#    print(ig_G.nodes()[node]["drug"])
ig_gcn_only_sorted_betweenness[:10], ig_gcn_bert_sorted_betweenness[:10], shap_gcn_only_sorted_betweenness[:10], shap_gcn_bert_sorted_betweenness[:10]

([(1912, 0.08575065973194793),
  (772, 0.081272796445265),
  (383, 0.06900527097090894),
  (775, 0.06750725002634889),
  (2068, 0.06093165828913155),
  (1877, 0.05867676317108321),
  (204, 0.056791230150669725),
  (1647, 0.05487891287334047),
  (2164, 0.0522562732917682),
  (2038, 0.051903826295213805)],
 [(310, 0.07600097033660615),
  (772, 0.07543798818648234),
  (1140, 0.069865927732816),
  (254, 0.06746097812109268),
  (774, 0.06359944892288218),
  (1838, 0.056915372766650665),
  (428, 0.05636233281195324),
  (1835, 0.05533510241514245),
  (553, 0.05487115722637642),
  (58, 0.054646957651803295)],
 [(310, 0.06359063447335246),
  (1912, 0.060060695866688746),
  (772, 0.05950691241010887),
  (1948, 0.05901816194202229),
  (254, 0.054287339908621786),
  (2068, 0.04796928220864737),
  (2164, 0.04326888081076821),
  (1838, 0.03919153940649794),
  (1471, 0.03620785870446689),
  (2297, 0.03604630103963398)],
 [(1912, 0.06535457455630098),
  (1838, 0.06045372553191939),
  (2465, 0.05039549

In [175]:
ig_gcn_only_communities = nx.community.greedy_modularity_communities(ig_gcn_only_G)
ig_gcn_only_modularity_dict = {}
for i, c in enumerate(ig_gcn_only_communities):
    for name in c:
        ig_gcn_only_modularity_dict[name] = i
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_modularity_dict, 'community')

ig_gcn_bert_communities = nx.community.greedy_modularity_communities(ig_gcn_bert_G)
ig_gcn_bert_modularity_dict = {}
for i, c in enumerate(ig_gcn_bert_communities):
    for name in c:
        ig_gcn_bert_modularity_dict[name] = i
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_modularity_dict, 'community')

shap_gcn_only_communities = nx.community.greedy_modularity_communities(shap_gcn_only_G)
shap_gcn_only_modularity_dict = {}
for i, c in enumerate(shap_gcn_only_communities):
    for name in c:
        shap_gcn_only_modularity_dict[name] = i
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_modularity_dict, 'community')

shap_gcn_bert_communities = nx.community.greedy_modularity_communities(shap_gcn_bert_G)
shap_gcn_bert_modularity_dict = {}
for i, c in enumerate(shap_gcn_bert_communities):
    for name in c:
        shap_gcn_bert_modularity_dict[name] = i
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_modularity_dict, 'community')

len(ig_gcn_only_communities), len(ig_gcn_bert_communities), len(shap_gcn_only_communities), len(shap_gcn_bert_communities)

(56, 49, 41, 39)

In [ ]:
([Counter([ig_gcn_only_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in ig_gcn_only_communities[:10]],
 [Counter([ig_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in ig_gcn_bert_communities[:10]],
 [Counter([shap_gcn_only_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in shap_gcn_only_communities[:10]],
 [Counter([shap_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in shap_gcn_bert_communities[:10]])

In [ ]:
([Counter([ig_gcn_only_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in ig_gcn_only_communities[:10]],
 [Counter([ig_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in ig_gcn_bert_communities[:10]],
 [Counter([shap_gcn_only_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in shap_gcn_only_communities[:10]],
 [Counter([shap_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in shap_gcn_bert_communities[:10]])

In [183]:
train_count = 0
val_count = 0
test_count = 0

for node in ig_gcn_bert_G.nodes():
    if node not in test_idx: 
        continue
    for n in ig_gcn_bert_G.adj[node]:
        if n in train_idx or n in val_idx:
            train_count += 1
        #elif n in val_idx:
        #    val_count += 1
        else:
            test_count += 1

s = train_count + val_count + test_count
train_count/s, val_count/s, test_count/s

(0.6767608029942157, 0.0, 0.3232391970057843)

In [184]:
train_count = 0
val_count = 0
test_count = 0

for node in shap_gcn_bert_G.nodes():
    if node not in test_idx: 
        continue
    for n in shap_gcn_bert_G.adj[node]:
        if n in train_idx or n in val_idx:
            train_count += 1
        #elif n in val_idx:
        #    val_count += 1
        else:
            test_count += 1

s = train_count + val_count + test_count
train_count/s, val_count/s, test_count/s

(0.6710191622859081, 0.0, 0.3289808377140919)